In [1]:
import time
import warnings
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display
import ipywidgets as widgets
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import lightkurve as lk
from lightkurve import search_lightcurve
import warnings
from astropy.units import UnitsWarning


/Users/wayfinder/Code/fault-in-our-stars/.env/lib/python3.13/site-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [2]:
import pandas as pd
from tqdm.auto import tqdm
import lightkurve as lk

def fetchStitchedLC(KIC):
    """
    Given a KIC, fetch its light curve data using Lightkurve.
    Returns: stitched LC object.
    """
    search = lk.search_lightcurve(
        f"KIC {KIC}",
        mission="Kepler",
        author="Kepler",
        cadence="long"
    )
    return search.download_all().stitch().remove_nans()

def getLCArrays(kic):
    lc = fetchStitchedLC(kic)

    return pd.Series({
        "time": lc.time.value,
        "flux": lc.flux.value,
        "flux_err": lc.flux_err.value if lc.flux_err is not None else None,
        "n_points": len(lc),
    })

def getLCArrays_safe(kic):
    try:
        return getLCArrays(kic)
    except Exception:
        return pd.Series({
            "time": None,
            "flux": None,
            "flux_err": None,
            "n_points": 0,
        })

def add_lightcurve_features_in_batches(df, batch_size=100):
    """
    df must be indexed by KIC.
    Processes rows in batches and appends light-curve features.
    """
    kics = df.index.to_list()
    out = []

    for start in tqdm(range(0, len(kics), batch_size), desc="Processing batches"):
        batch_kics = kics[start:start + batch_size]
        batch_features = [getLCArrays_safe(kic) for kic in tqdm(batch_kics, leave=False, desc="KICs in batch")]
        out.extend(batch_features)

    feature_df = pd.DataFrame(out, index=df.index)
    return df.join(feature_df)



In [3]:
df = pd.read_csv(r"../assets/data/kepler-eclipsing-binary-catalog.csv")
df = df.drop(columns=["Unnamed: 11"])
# df = df.head(150)
df = df.set_index("KIC")


In [4]:
df


,period,period_err,bjd0,bjd0_err,morph,GLon,GLat,kmag,Teff,SC
KIC,,,,,,,,,,
3863594,0.053268,0.0,55000.000000,0.004327,0.79,-1.0000,-1.0000,-1.000,-1.0,False
10417986,0.073731,0.0,55000.027476,0.004231,0.99,81.0390,11.0820,9.128,-1.0,True
8912468,0.094838,0.0,54953.576945,0.005326,0.98,80.1095,7.8882,11.751,6194.0,False
8758716,0.107205,0.0,54953.672989,0.006197,1.00,77.7478,11.6565,13.531,-1.0,False
10855535,0.112782,0.0,54964.629315,0.006374,0.99,79.3949,15.9212,13.870,7555.0,False
...,...,...,...,...,...,...,...,...,...,...
9408440,989.985000,-1.0,55346.365980,0.096130,0.00,78.5607,12.2615,13.199,5688.0,False
8054233,1058.000000,-1.0,54751.806288,0.968052,0.03,78.6142,7.7321,11.783,4733.0,False
7672940,1064.270000,-1.0,54977.092960,0.089646,0.00,74.5296,14.6136,12.328,-1.0,False


**This method got stuck at batch 53/59 (90%) on batch 12/50 (24%) after 312 mins.**

In [5]:
# Example:
df_with_lc = add_lightcurve_features_in_batches(df, batch_size=50)


Processing batches:   0%|          | 0/59 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

...

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KICs in batch:   0%|          | 0/50 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# If KIC is the index:
# df_with_lc = download_lightcurves(df, max_workers=8)

# If KIC is in a column:
# df_with_lc = parallel_fetch_lightcurves(df, max_workers=8, kic_col="KIC")


In [ ]:
df_with_lc


In [ ]:
df_with_lc.to_json("../assets/data/keb-lcs.json")
